# Arrhythmia Detection 1D-CNN — Phase 1 & 2 In-Domain Training (Kaggle GPU)

Fully **self-contained** notebook (no `src/` dependency) that trains the
**softmax + native (all-class)** baselines on the mounted **"ECG Dataset"**
(PTB-XL / Chapman tensors at 100 Hz and 500 Hz, raw and cleaned).

**What this notebook does — and only does:**

1. Auto-detects the dataset root under `/kaggle/input` (works whatever the slug).
2. Picks the active folders from `DATA_SELECTION` + `RUN_PHASES` (set in the
   *Pick folders* cell). Empty folders are skipped automatically.
3. Trains **one model per dataset × folder** (Phase 1 = RAW, Phase 2 = CLEANED).
4. Saves per run under `/kaggle/working/results/phase1_2/<experiment_id>/`:
   `best_model.keras`, `metrics.csv`, `classification_report.txt`,
   `training_history.csv`, `predictions.npz`, `experiment_metadata.json`,
   `confusion_matrix*.png`, `misclassified/` plots.
5. **Evaluates inline** — right after each run finishes, the notebook renders the
   confusion matrix (counts + normalized) and the first misclassified ECG samples
   (3-lead clinical panels) so you can eyeball the result immediately.
6. Writes `run_summary.csv` and prints the full output tree — every artifact is
   saved directly under `/kaggle/working/results/phase1_2/` (no zip needed; grab
   files from Kaggle's **Files** panel).

**What runs locally later (NOT here):** statistical tests, Grad-CAM,
cross-dataset evaluation. Those consume the saved `*.keras` / `*.npz` / CSVs with
the local repo (`src/evaluation/*`, `src/experiments/*`). No **stress test** and no
**(macro) AUROC** are computed anywhere in the pipeline.

**Loading a saved model later** (Keras 3):

    from model_factory import StochasticDepth   # or define it locally
    model = tf.keras.models.load_model(
        "best_model.keras", compile=False,
        custom_objects={"StochasticDepth": StochasticDepth})

Use `compile=False`: the embedded custom loss is a closure that Keras 3 cannot
relocate, and predictions do not need the compile state.

Expected dataset root layout:

    ptbxl_{raw,clean}_{100,500}hz/   chapman_{raw,clean}_{100,500}hz/
    manifest_ptbxl.csv               manifest_chapman.csv


In [ ]:
import os, sys, re, gc, json, random, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, balanced_accuracy_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)


def _find_manifest(name):
    for root, _dirs, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    return None


MAN_PTBXL = _find_manifest("manifest_ptbxl.csv")
MAN_CHAPMAN = _find_manifest("manifest_chapman.csv")
if MAN_PTBXL is None and MAN_CHAPMAN is None:
    MAN_PTBXL = "/kaggle/input/ekg-dataset/manifest_ptbxl.csv"
    MAN_CHAPMAN = "/kaggle/input/ekg-dataset/manifest_chapman.csv"
DATA_ROOT = os.path.dirname(MAN_PTBXL or MAN_CHAPMAN)
print("Dataset root:", DATA_ROOT)

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print(f"[Env] OS={platform.system()} | Python={platform.python_version()} "
      f"| TF={tf.__version__} | GPUs={[g.name for g in gpus] or 'CPU'}")

OUT_ROOT = "/kaggle/working/results/phase1_2"
os.makedirs(OUT_ROOT, exist_ok=True)

_K = ["ptbxl_raw_100hz", "ptbxl_clean_100hz", "ptbxl_raw_500hz", "ptbxl_clean_500hz",
      "chapman_raw_100hz", "chapman_clean_100hz", "chapman_raw_500hz", "chapman_clean_500hz"]
SUB_FOLDERS = {k: os.path.join(DATA_ROOT, k) for k in _K}
FOLDER_FS = {k: 500 if "500hz" in k else 100 for k in _K}
TARGET_LEN = {100: 1000, 500: 5000}
NUM_LEADS = 3

In [ ]:
# ---------- Manifests: native (all-class) labels + splits ----------

def build_df(dataset):
    path = MAN_PTBXL if dataset == "PTBXL" else MAN_CHAPMAN
    if not path or not os.path.exists(path):
        raise FileNotFoundError(
            f"[{dataset}] manifest not found at {path}. Attach the ECG Dataset "
            "('ECG Dataset') or check the dataset slug.")
    df = pd.read_csv(path)
    df["filename_npy"] = df["filename_npy"].astype(str)
    if "native_label" in df.columns:
        df["label"] = df["native_label"].astype(str)
    else:  # fallback: Chapman diagnostic string -> primary acronym/code
        df["label"] = df["diagnostic_string"].map(
            lambda s: (None if pd.isna(s) else
                       [c.strip() for c in str(s).split(",") if c.strip()][:1][0]))
    df = df[df["label"].notna() & ~df["label"].isin(["", "None", "nan"])]
    # ---- PHASE 4 (3-lead clinically detectable subset) --------------------
    # Active only when PHASE4_ENABLED = True (set in the 'Pick folders' cell).
    # Guarded with globals() so running cells out of order never NameErrors.
    if globals().get("PHASE4_ENABLED") and globals().get("PHASE4_ALLOWLIST", {}).get(dataset):
        allow = PHASE4_ALLOWLIST[dataset]
        df = df[df["label"].isin(allow)]
        print(f"[PHASE4] {dataset} -> {len(df)} rows | classes={allow}")
    # -----------------------------------------------------------------------
    return df.reset_index(drop=True)


def assign_splits(df, dataset, min_count=8):
    if dataset == "PTBXL":  # native stratified folds
        fold = df["strat_fold"].astype(int)
        return df.assign(split=np.where(fold <= 8, "train",
                                        np.where(fold == 9, "val", "test")))
    counts = df["label"].value_counts()
    rare = counts[counts < min_count].index.tolist()  # cannot stratify
    if rare:
        df = df[~df["label"].isin(rare)].reset_index(drop=True)
        print(f"[CHAPMAN] dropped {len(rare)} rare classes (< {min_count}): {rare[:8]}")
    train, rest = train_test_split(df, test_size=0.30,
                                   stratify=df["label"], random_state=42)
    val, test = train_test_split(rest, test_size=0.50,
                                 stratify=rest["label"], random_state=42)
    return pd.concat([train.assign(split="train"), val.assign(split="val"),
                      test.assign(split="test")]).reset_index(drop=True)


def load_dataset(df, folder_key):
    # .npy arrays + one-hot labels for train/val/test under one class order.
    class_names = sorted(df["label"].astype(str).unique())
    lookup = {n: i for i, n in enumerate(class_names)}
    folder = SUB_FOLDERS[folder_key]
    out = {}
    for split in ("train", "val", "test"):
        sub = df[df["split"] == split]
        X, lab = [], []
        for row in sub.itertuples(index=False):
            path = os.path.join(folder, row.filename_npy)
            if not os.path.exists(path):
                continue
            try:
                X.append(np.load(path).astype(np.float32))
                lab.append(str(row.label))
            except Exception:
                continue
        y = np.zeros((len(lab), len(class_names)), dtype=np.float32)
        for i, name in enumerate(lab):
            y[i, lookup[name]] = 1.0
        out[split] = (np.asarray(X, np.float32), y)
    return out, class_names


def class_weights(y_train):
    labels = np.argmax(y_train, axis=1)
    classes = np.unique(labels)
    w = compute_class_weight("balanced", classes=classes, y=labels)
    return {int(c): float(x) for c, x in zip(classes, w)}


# ---------- tf.data pipeline (light online augmentation, safe ranges) ----------

def _augment(signal, label):
    # Graph-safe online augmentation: explicit tf.cond only (no AutoGraph).
    def noop():
        return signal, label

    def lead_drop():
        idx = tf.random.uniform([], 0, NUM_LEADS, tf.int32)
        return signal * (1.0 - tf.one_hot(idx, NUM_LEADS, dtype=tf.float32)), label

    def add_noise():
        return signal + tf.random.normal(tf.shape(signal), 0.0, 0.003), label

    def gain_scale():
        return signal * tf.random.uniform([], 0.99, 1.01), label

    s, l = tf.cond(tf.random.uniform([]) < 0.03, lead_drop, noop)
    s, l = tf.cond(tf.random.uniform([]) < 0.20, add_noise,
                   lambda: (s, l))
    s, l = tf.cond(tf.random.uniform([]) < 0.15, gain_scale,
                   lambda: (s, l))
    return s, l


def create_dataset(X, y, batch_size, is_training=False, use_augmentation=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (X.astype(np.float32), y.astype(np.float32)))
    if is_training:
        ds = ds.shuffle(min(len(X), 100000), reshuffle_each_iteration=True)
        if use_augmentation:
            ds = ds.map(_augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
# ---------- 1D-CNN backbone (Stochastic Depth residual + SE support) ----------

@tf.keras.utils.register_keras_serializable(package="model_factory")
class StochasticDepth(tf.keras.layers.Layer):
    def __init__(self, survival_probability=1.0, **kwargs):
        super().__init__(**kwargs)
        self.survival_probability = survival_probability

    def call(self, x, residual, training=None):
        if training:
            p = tf.cast(tf.random.uniform([]) < self.survival_probability,
                        tf.float32)
            x = (p * x) / self.survival_probability
        return x + residual

    def get_config(self):
        c = super().get_config()
        c["survival_probability"] = self.survival_probability
        return c


L2 = tf.keras.regularizers.l2


def _conv(x, filters, kernel, dilation, separable):
    if separable:
        return tf.keras.layers.SeparableConv1D(
            filters, kernel, padding="same", dilation_rate=dilation,
            depthwise_regularizer=L2(3e-4),
            pointwise_regularizer=L2(3e-4))(x)
    return tf.keras.layers.Conv1D(
        filters, kernel, padding="same", dilation_rate=dilation,
        kernel_regularizer=L2(3e-4))(x)


def residual_block(x, filters, kernel, dilation, dropout=0.15,
                   separable=True, sd_rate=0.0):
    shortcut = x
    y = _conv(x, filters, kernel, dilation, separable)
    y = tf.keras.layers.BatchNormalization()(y)
    y = tf.keras.layers.Activation("relu")(y)
    y = _conv(y, filters, kernel, dilation, separable)
    y = tf.keras.layers.BatchNormalization()(y)
    if shortcut.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv1D(filters, 1, padding="same")(shortcut)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)
    y = StochasticDepth(1.0 - sd_rate)(y, shortcut)
    y = tf.keras.layers.Activation("relu")(y)
    return tf.keras.layers.SpatialDropout1D(dropout)(y)


def build_cnn(filters, kernels, dilations, input_shape, num_classes,
              separable=False, sd_rate=0.0):
    inputs = tf.keras.layers.Input(shape=input_shape)
    x = inputs
    for i, (f, k, d) in enumerate(zip(filters, kernels, dilations)):
        x = residual_block(x, f, k, d, dropout=0.15, separable=separable,
                           sd_rate=sd_rate * (i + 1) / len(filters))
        if i < 3:
            x = tf.keras.layers.MaxPooling1D(2, 2, padding="same")(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(128, use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.Dropout(0.20)(x)
    out = tf.keras.layers.Dense(num_classes, activation="softmax",
                                name="softmax_output")(x)
    return tf.keras.models.Model(inputs, out, name="ECG_1DCNN_P1P2")


# ---------- stable focal loss (multiclass) ----------

def focal_loss(gamma=2.0, alpha=1.0, label_smoothing=0.0):
    def loss_fn(y_true, y_pred):
        if label_smoothing > 0.0:
            n = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1.0 - label_smoothing) + label_smoothing / n
        eps = 1e-7
        p = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        ce = tf.reduce_sum(-y_true * tf.math.log(p), axis=-1)
        pt = tf.reduce_sum(y_true * p, axis=-1)
        return tf.reduce_mean(alpha * tf.pow(1.0 - pt, gamma) * ce)
    return loss_fn

In [ ]:
# ---------- Metrics (no AUROC / no stress test) + one training run ----------

def compute_metrics(y_true, y_pred, y_prob, class_names, exp_id):
    d = {"Experiment": exp_id,
         "Accuracy": float(np.mean(y_true == y_pred)),
         "Balanced_Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
         "Macro_Precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
         "Macro_Recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
         "Macro_F1": float(f1_score(y_true, y_pred, average="macro", zero_division=0))}
    rep = classification_report(y_true, y_pred, labels=list(range(len(class_names))),
                                target_names=class_names, output_dict=True,
                                zero_division=0)
    for i, name in enumerate(class_names):
        if name in rep:
            d[f"Sens_{name}"] = float(rep[name]["recall"])
            d[f"Precision_{name}"] = float(rep[name]["precision"])
            d[f"F1_{name}"] = float(rep[name]["f1-score"])
    return d


def plot_confusion_matrix(y_true, y_pred, class_names, exp_dir):
    labels = list(range(len(class_names)))
    for norm, tag, fname in ((False, "Confusion Matrix", "confusion_matrix.png"),
                             (True, "Confusion Matrix (Normalized)",
                              "confusion_matrix_normalized.png")):
        cm = confusion_matrix(y_true, y_pred, labels=labels,
                              normalize="true" if norm else None)
        fig, ax = plt.subplots(figsize=(0.34 * len(class_names) + 5,
                                        0.34 * len(class_names) + 5))
        ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=class_names).plot(
            ax=ax, cmap="Blues", xticks_rotation="vertical",
            colorbar=True, values_format=".2f" if norm else "d")
        ax.set_title(tag, fontsize=12, fontweight="bold")
        plt.tight_layout()
        fig.savefig(os.path.join(exp_dir, fname), dpi=150)
        plt.show()
        plt.close(fig)


def plot_misclassified(X, y_true, y_pred, class_names, exp_dir, max_samples=6):
    mis = np.where(y_true != y_pred)[0]
    if len(mis) == 0:
        print("  No misclassified samples in the test split.")
        return
    out = os.path.join(exp_dir, "misclassified")
    os.makedirs(out, exist_ok=True)
    for i, idx in enumerate(mis[:max_samples]):
        sig = X[idx]
        true_cls = class_names[y_true[idx]]
        pred_cls = class_names[y_pred[idx]]
        safe = lambda s: re.sub(r"[^\w\-]+", "_", s)
        n = sig.shape[-1]
        fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(14, 2 * n), sharex=True)
        if n == 1:
            axes = [axes]
        for ch in range(n):
            axes[ch].plot(sig[:, ch], color="#1c1c1e", linewidth=1.2)
            axes[ch].set_ylabel(f"Lead {ch+1}", fontsize=10, fontweight="bold")
            axes[ch].grid(True, linestyle="--", alpha=0.5, color="#d2d2d7")
        fig.suptitle(f"ERROR #{i} | True: {true_cls} -> Pred: {pred_cls}",
                     fontsize=12, fontweight="bold", color="#ff453a")
        plt.tight_layout()
        fig.savefig(os.path.join(out, f"{i:02d}_{safe(true_cls)}_to_{safe(pred_cls)}.png"),
                    dpi=150)
        plt.show()
        plt.close(fig)


def train_one(dataset, folder_key, **hp):
    fs = FOLDER_FS[folder_key]
    input_shape = (TARGET_LEN[fs], NUM_LEADS)
    exp_id = ("RAW" if "raw" in folder_key else "CLEAN") + f"_{dataset}_3L"
    exp_dir = os.path.join(OUT_ROOT, exp_id)
    os.makedirs(exp_dir, exist_ok=True)

    print("\n" + "=" * 70)
    print(f"[EXPERIMENT] {exp_id} | {dataset} @ {folder_key} ({fs} Hz, {input_shape})")
    print("=" * 70)

    df = build_df(dataset)
    df = assign_splits(df, dataset)
    splits, class_names = load_dataset(df, folder_key)
    Xtr, ytr = splits["train"]; Xva, yva = splits["val"]; Xte, yte = splits["test"]
    print(f"train={len(Xtr)} val={len(Xva)} test={len(Xte)} | classes={len(class_names)}")
    tr_counts = np.bincount(np.argmax(ytr, axis=1), minlength=len(class_names))
    for name, c in zip(class_names, tr_counts):
        print(f"   {name:<10} {c}")

    model = build_cnn(hp["filters"], hp["kernels"], hp["dilations"],
                      input_shape, len(class_names))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp["lr"]),
                  loss=focal_loss(hp["gamma"], hp["alpha"], hp["label_smoothing"]),
                  metrics=["accuracy"])

    model_path = os.path.join(exp_dir, "best_model.keras")
    callbacks = [
        tf.keras.callbacks.EarlyStopping("val_loss", patience=hp["patience"],
                                         restore_best_weights=True, verbose=0),
        tf.keras.callbacks.ModelCheckpoint(model_path, save_best_only=True, verbose=0),
        tf.keras.callbacks.CSVLogger(os.path.join(exp_dir, "training_log.csv"))]

    train_ds = create_dataset(Xtr, ytr, hp["batch_size"], is_training=True,
                              use_augmentation=hp["use_augmentation"])
    val_ds = create_dataset(Xva, yva, hp["batch_size"])
    hist = model.fit(train_ds, validation_data=val_ds, epochs=hp["epochs"],
                     callbacks=callbacks, class_weight=class_weights(ytr),
                     verbose=1)

    best = tf.keras.models.load_model(
        model_path, compile=False,
        custom_objects={"StochasticDepth": StochasticDepth})
    y_prob = best.predict(Xte, batch_size=hp["batch_size"], verbose=0)
    y_pred = np.argmax(y_prob, axis=1).astype(np.int32)
    y_true = np.argmax(yte, axis=1).astype(np.int32)

    metrics = compute_metrics(y_true, y_pred, y_prob, class_names, exp_id)
    metrics.update({"Scheme": "softmax", "Label_Scheme": "native",
                    "Dataset": dataset, "Folder": folder_key,
                    "Sampling_Rate_Hz": fs, "Signal_Len": input_shape[0],
                    "Num_Classes": len(class_names),
                    "Epochs_Done": int(len(hist.history["loss"])),
                    "Model_Size_MB": round(os.path.getsize(model_path) / 1e6, 2),
                    "Total_Params": int(best.count_params())})
    pd.DataFrame([metrics]).to_csv(os.path.join(exp_dir, "metrics.csv"), index=False)

    rep_str = classification_report(y_true, y_pred, labels=list(range(len(class_names))),
                                    target_names=class_names, digits=4, zero_division=0)
    with open(os.path.join(exp_dir, "classification_report.txt"),
              "w", encoding="utf-8") as f:
        f.write(rep_str)

    pd.DataFrame(hist.history).to_csv(
        os.path.join(exp_dir, "training_history.csv"), index=False)
    np.savez_compressed(os.path.join(exp_dir, "predictions.npz"),
                        y_true=y_true, y_pred=y_pred, y_prob=y_prob,
                        class_names=np.array(class_names))

    print("[PLOT] confusion matrix + misclassified samples ...")
    plot_confusion_matrix(y_true, y_pred, class_names, exp_dir)
    plot_misclassified(Xte, y_true, y_pred, class_names, exp_dir, max_samples=6)

    meta = {"experiment_id": exp_id, "dataset": dataset, "folder": folder_key,
            "sampling_rate_hz": fs, "signal_len": input_shape[0],
            "label_scheme": "native", "num_classes": len(class_names),
            "class_names": class_names, "input_shape": list(input_shape),
            "hyperparameters": {k: hp.get(k) for k in
                                ("filters", "kernels", "dilations", "epochs",
                                 "batch_size", "lr", "gamma", "alpha",
                                 "label_smoothing", "use_augmentation")},
            "environment": {"os": platform.system(),
                            "python": platform.python_version(),
                            "tensorflow": tf.__version__,
                            "gpu": [g.name for g in tf.config.list_physical_devices("GPU")]},
            "status": "completed"}
    with open(os.path.join(exp_dir, "experiment_metadata.json"),
              "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print(f"[DONE {exp_id}] Macro_F1={metrics['Macro_F1']:.4f} "
          f"BalAcc={metrics['Balanced_Accuracy']:.4f} | {exp_dir}")
    tf.keras.backend.clear_session()
    gc.collect()
    return metrics


# ---------- plan runner + output listing ----------

def _summary_row(metrics):
    # run_summary.csv keeps macro/scalar columns only; per-class metrics
    # stay in each experiment's metrics.csv (so PTBXL + CHAPMAN rows align).
    return {k: v for k, v in metrics.items()
            if not (k.startswith("Sens_") or k.startswith("Precision_")
                    or k.startswith("F1_"))}


def run_plans(plans, hp):
    rows = []
    for dataset, folder_key in plans:
        folder = SUB_FOLDERS[folder_key]
        if not os.path.isdir(folder) or not os.listdir(folder):
            print(f"SKIP (empty folder): {folder_key}")
            continue
        exp_id = ("RAW" if "raw" in folder_key else "CLEAN") + f"_{dataset}_3L"
        try:
            rows.append(_summary_row(train_one(dataset, folder_key, **hp)))
        except Exception as e:
            print(f"FAILED {exp_id}: {e}")
            rows.append({"Experiment": exp_id, "error": str(e)})
    summary = pd.DataFrame(rows)
    summary.to_csv(os.path.join(OUT_ROOT, "run_summary.csv"), index=False)
    print("\n" + "=" * 70)
    print("RUN SUMMARY")
    print("=" * 70)
    print(summary.to_string(index=False))
    return summary


def show_output_tree():
    print("\nSaved artifacts (download straight from the Files panel — no zip needed):")
    n_files = 0
    total = 0
    for root, _dirs, files in os.walk(OUT_ROOT):
        for name in files:
            full = os.path.join(root, name)
            size = os.path.getsize(full)
            n_files += 1
            total += size
            print(f"  {os.path.relpath(full, OUT_ROOT):<72} {size/1024:9,.1f} KB")
    print(f"\n{n_files} files | {total/1024/1024:,.2f} MB under {OUT_ROOT}")

In [ ]:
# ---------------------------------------------------------------------
# PICK FOLDERS — edit these, then run the last cell.
# ---------------------------------------------------------------------
DATA_SELECTION = "500Hz"                       # "500Hz" | "100Hz"
RUN_PHASES = ["Phase1_Raw", "Phase2_Cleaned"]  # Phase1 = raw, Phase2 = cleaned

DATASETS = ["PTBXL", "CHAPMAN"]
PHASE_KINDS = {"Phase1_Raw": "raw", "Phase2_Cleaned": "clean"}

# FAST_MODE means a quick smoke test; False uses the reference configuration.
FAST_MODE = True

if FAST_MODE:
    HP = dict(epochs=5, batch_size=64, lr=3e-4, gamma=2.0, alpha=1.0,
              label_smoothing=0.0, use_augmentation=False, patience=3,
              filters=[32, 64, 128, 128, 256],
              kernels=[15, 11, 7, 5, 3], dilations=[1, 2, 4, 8, 16])
else:
    HP = dict(epochs=35, batch_size=64, lr=3e-4, gamma=2.0, alpha=1.0,
              label_smoothing=0.0, use_augmentation=False, patience=10,
              filters=[64, 128, 256, 256, 512],
              kernels=[15, 11, 7, 5, 3], dilations=[1, 2, 4, 8, 16])

_FS = {"500Hz": "500hz", "100Hz": "100hz"}[DATA_SELECTION]
assert all(p in PHASE_KINDS for p in RUN_PHASES), list(PHASE_KINDS)

PLANS = []
for ph in RUN_PHASES:
    for ds in DATASETS:
        prefix = "ptbxl" if ds == "PTBXL" else "chapman"
        PLANS.append((ds, f"{prefix}_{PHASE_KINDS[ph]}_{_FS}"))

print(f"DATA_SELECTION = {DATA_SELECTION} | phases = {RUN_PHASES} | FAST_MODE = {FAST_MODE}")
print(f"HP = {HP}")
for ds, fk in PLANS:
    folder = SUB_FOLDERS[fk]
    n = len(os.listdir(folder)) if os.path.isdir(folder) else 0
    print(f"  {ds:<7} {fk:<24} {n} files {'OK' if n else 'EMPTY'}")

# =========================================================================
# PHASE 4 — CLINICALLY DETECTABLE 3-LEAD SUBSET
# =========================================================================
# docs/dataset-label-map.md, "3-Lead (I, II, III)" row:
#   all main arrhythmias + inferior/lateral MI + AV blocks + WPW.
# The names below are mapped to the native labels that actually exist in
# each dataset manifest (PTB-XL = SCP codes, Chapman = rhythm acronyms).
#
# Set PHASE4_ENABLED = True to retrain only on this subset (Phase 4).
# Note tiny PTB-XL classes: 2AVB=3 and 3AVB=4 records in the manifest.
PHASE4_ENABLED = False
PHASE4_ALLOWLIST = {
    "PTBXL": [
        # AFIB, AFLT, SVT            -> 'PSVT' is PTB-XL's SVT code
        "AFIB", "AFLT", "PSVT",
        # AV blocks (1st/2nd/3rd degree)
        "1AVB", "2AVB", "3AVB",
        # WPW + Inferior MI + Lateral MI
        "WPW", "IMI", "LMI",
    ],
    "CHAPMAN": [
        # Chapman is a rhythm dataset; only its main rhythms carry native
        # labels (no AVB / WPW / MI codes exist as native classes there).
        "AFIB", "AFLT", "AT", "AVRT", "SB", "SI", "SR", "ST", "SVT",
    ],
}

In [ ]:
run_summary = run_plans(PLANS, HP)
show_output_tree()

print("\nDone. Output is saved directly under /kaggle/working/results/phase1_2/ "
      "— download the files you need from the Files panel and run the "
      "post-analysis modules locally (src/).")